In [13]:
# Install Groq SDK if you want to test the LLM agent directly inside Colab
!pip install -q groq scikit-learn pandas numpy joblib

In [14]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [15]:
# Streaming dataset directly from public URL
url = "https://raw.githubusercontent.com/guipsamora/pandas_exercises/master/07_Visualization/Online_Retail/Online_Retail.csv"

print("Fetching dataset from direct URL...")
df = pd.read_csv(url, encoding="latin1")

print(f"Dataset shape: {df.shape}")
df.head()

Fetching dataset from direct URL...
Dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [16]:
# Drop missing CustomerIDs and invalid transactions
df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Split data: First 9 months = Features (RFM), Final 3 months = Target (Future Spend)
cutoff_date = df['InvoiceDate'].min() + pd.DateOffset(months=9)

obs_df = df[df['InvoiceDate'] <= cutoff_date]
future_df = df[df['InvoiceDate'] > cutoff_date]

print("Data successfully cleaned and split into observation/target windows.")

/tmp/ipykernel_1527/3802323494.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


Data successfully cleaned and split into observation/target windows.


In [17]:
# RFM features from observation window
features = obs_df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (cutoff_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                # Frequency
    'TotalAmount': ['sum', 'mean']                         # Monetary & Avg Order Value
}).reset_index()

features.columns = ['CustomerID', 'recency_days', 'frequency_orders', 'hist_monetary', 'avg_order_value']

# Target: Actual future spend in the remaining 3 months
target = future_df.groupby('CustomerID')['TotalAmount'].sum().reset_index()
target.columns = ['CustomerID', 'future_clv']

# Merge features and target
customer_df = pd.merge(features, target, on='CustomerID', how='left').fillna(0)
customer_df.head()

,CustomerID,recency_days,frequency_orders,hist_monetary,avg_order_value,future_clv
0,12346,225,1,77183.60,77183.600000,0.00
1,12347,29,5,2790.86,22.506935,1519.14
2,12348,148,3,1487.24,53.115714,310.00
3,12350,210,1,334.40,19.670588,0.00
4,12352,162,5,1561.81,41.100263,944.23


In [18]:
X = customer_df[['recency_days', 'frequency_orders', 'hist_monetary', 'avg_order_value']]
y = customer_df['future_clv']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Save the updated .joblib files directly in Colab
joblib.dump(model, "clv_model.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(list(X.columns), "feature_names.joblib")

print("Updated artifacts exported successfully!")

Updated artifacts exported successfully!


In [19]:
# Save artifacts for app.py and agent.py
joblib.dump(model, "clv_model.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(list(X.columns), "feature_names.joblib")

print("Saved files: 'clv_model.joblib', 'scaler.joblib', 'feature_names.joblib'")

Saved files: 'clv_model.joblib', 'scaler.joblib', 'feature_names.joblib'
